In [1]:
import dotenv
import os

dotenv.load_dotenv(dotenv_path='.env')

True

In [2]:
import pyarrow as pa
import pandas
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.exceptions import NamespaceAlreadyExistsError
from pyiceberg.transforms import IdentityTransform, DayTransform

/Users/jhonsfran/repos/unprice/internal/lakehouse/lakenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [29]:
# 1. Connect to R2 Data Catalog

# Define catalog connection details
ENV = "dev"
CATALOG = "unprice-lakehouse-" + ENV
ACCOUNT_ID  = os.environ['ACCOUNT_ID']
CATALOG_URI = "https://catalog.cloudflarestorage.com/" + ACCOUNT_ID + "/" + CATALOG
TOKEN       = os.environ['TOKEN']
WAREHOUSE   = ACCOUNT_ID + "_" + CATALOG

# Connect to R2 Data Catalog
catalog = RestCatalog(
    name="unprice",
    warehouse=WAREHOUSE,
    uri=CATALOG_URI,
    token=TOKEN,
)

# list namespaces
print(catalog.list_namespaces())

[('lakehouse',)]


In [17]:
# drop tables
# catalog.drop_table('lakehouse.metadata')
# catalog.drop_table('lakehouse.entitlement_snapshot')
# catalog.drop_table('lakehouse.usage')
# catalog.drop_table('lakehouse.verification')

In [18]:
# List tables in the "default" namespace
tables = catalog.list_tables(namespace='lakehouse')
print(tables)

[('lakehouse', 'usage'), ('lakehouse', 'entitlement_snapshot'), ('lakehouse', 'verification'), ('lakehouse', 'metadata')]


In [19]:
# Load and print schema for every table in the lakehouse namespace
tables = catalog.list_tables(namespace='lakehouse')
for table_id in tables:
    table = catalog.load_table(table_id)
    print(f"--- {table_id[0]}.{table_id[1]} ---")
    print(table.schema())
    print()

--- lakehouse.usage ---
table {
  1: __ingest_ts: required timestamp
  2: id: required string
  3: event_date: required timestamp
  4: request_id: required string
  5: project_id: required string
  6: customer_id: required string
  7: timestamp: required long
  8: idempotence_key: required string
  9: entitlement_id: required string
  10: feature_slug: required string
  11: schema_version: required int
  12: usage: optional double
  13: allowed: optional boolean
  14: deleted: optional long
  15: meta_id: optional string
  16: country: optional string
  17: region: optional string
  18: action: optional string
  19: key_id: optional string
  20: unit_of_measure: optional string
  21: cost: optional double
  22: rate_amount: optional double
  23: rate_currency: optional string
}

--- lakehouse.entitlement_snapshot ---
table {
  1: __ingest_ts: required timestamp
  2: id: required string
  3: event_date: required timestamp
  4: project_id: required string
  5: customer_id: required strin

In [30]:
# List and peek contents of every table in the lakehouse namespace
import pandas as pd

tables = catalog.list_tables(namespace='lakehouse')
MAX_ROWS = 100  # rows to show per table

for table_id in tables:
    name = f"{table_id[0]}.{table_id[1]}"
    table = catalog.load_table(table_id)
    scan = table.scan(limit=MAX_ROWS)
    df = scan.to_pandas()
    print(f"\n=== {name} (showing up to {MAX_ROWS} rows) ===")
    if df.empty:
        print("(no rows)")
    else:
        display(df)


=== lakehouse.usage (showing up to 100 rows) ===


,__ingest_ts,id,event_date,request_id,project_id,customer_id,timestamp,idempotence_key,entitlement_id,feature_slug,...,deleted,meta_id,country,region,action,key_id,unit_of_measure,cost,rate_amount,rate_currency
0,2026-02-16 23:03:42,01KHMB3XT4WX0D10AC3VM958FG,2026-02-16,req_11Tw1B1uPRMfNX4suZXUx9,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283019584,123e4567-e89b-12d3-a456-426614174000:177128301...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
1,2026-02-16 23:03:42,01KHMB3XM0D3QCBM8APA4KWQDT,2026-02-16,req_11Tw1B14HgD7CTbLrPpnq6,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283019386,123e4567-e89b-12d3-a456-426614174000:177128301...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
2,2026-02-16 23:03:42,01KHMB3XDMQMMB75JGDZMPFCXJ,2026-02-16,req_11Tw1AzBx28QBuzmUcEhNy,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283019183,123e4567-e89b-12d3-a456-426614174000:177128301...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
3,2026-02-16 23:03:42,01KHMB3X5HSG3ZZNYPEVF06WWZ,2026-02-16,req_11Tw1Ay6DhuvaqjSULmL8y,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283018926,123e4567-e89b-12d3-a456-426614174000:177128301...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
4,2026-02-16 23:03:42,01KHMB3WSSAHRKW6VNCQCF1JNK,2026-02-16,req_11Tw1AwUV711EV33sfPDu7,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283018548,123e4567-e89b-12d3-a456-426614174000:177128301...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,2026-02-16 23:03:26,01KHMB3BAKV9HJ9J0JE85HDHYT,2026-02-16,req_11Tw19cTiKk79yrhz2onsD,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283000535,123e4567-e89b-12d3-a456-426614174000:177128300...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
74,2026-02-16 23:03:35,01KHMB3N3MSSKEKKEQREC2PY4Q,2026-02-16,req_11Tw1AMovwqAtHY2Z17549,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283010674,123e4567-e89b-12d3-a456-426614174000:177128301...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
75,2026-02-16 23:03:35,01KHMB3MXF69GYJ181PVJJ0QXZ,2026-02-16,req_11Tw1ALx648AKifWZVGY2m,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283010473,123e4567-e89b-12d3-a456-426614174000:177128301...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR
76,2026-02-16 23:03:35,01KHMB3MR2WWDDXQXVDW320F1E,2026-02-16,req_11Tw1ALDw2EHjtC8fXAebP,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1771283010303,123e4567-e89b-12d3-a456-426614174000:177128301...,ent_11TqNEG4SP8upafpUEmiqK,events,...,0,4055494417427953000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.0,0.0,EUR



=== lakehouse.entitlement_snapshot (showing up to 100 rows) ===


,__ingest_ts,id,event_date,project_id,customer_id,schema_version,timestamp,feature_slug,feature_type,unit_of_measure,reset_config,aggregation_method,merging_policy,limit,effective_at,expires_at,version,grants,metadata
0,2026-02-16 23:03:27,ent_11TqNEG4SP8upafpUEmiqK,2026-02-13,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1,2026-02-13 23:36:12.951,events,usage,units,"{""name"":""monthly"",""resetInterval"":""month"",""res...",sum,sum,1000,2026-02-13 23:36:12.320,NaT,sotumT+lUv/JLFyHYaCuMH0Zf+AxNKqe90AFAQBGaFU=,"[{""id"":""grnt_11TqNEDn26xX7Ut6Q7Baqn"",""type"":""s...","{""realtime"":false,""notifyUsageThreshold"":95,""o..."
1,2026-02-16 23:04:05,ent_11TqNEG2TKdxSGY6hW66MW,2026-02-13,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1,2026-02-13 23:36:12.943,customers,usage,units,"{""name"":""monthly"",""resetInterval"":""month"",""res...",sum,sum,100,2026-02-13 23:36:12.320,NaT,NN/jawHRQ91tcBRL4KpLE+sjoHCUnFjsyBInhVXL8Hw=,"[{""id"":""grnt_11TqNEDk33TR17axp8SA98"",""type"":""s...","{""overageStrategy"":""none"",""realtime"":false,""no..."



=== lakehouse.verification (showing up to 100 rows) ===


,__ingest_ts,id,event_date,project_id,entitlement_id,feature_slug,customer_id,request_id,timestamp,schema_version,...,region,meta_id,action,key_id,usage,remaining,unit_of_measure,cost,rate_amount,rate_currency
0,2026-02-16 23:01:15,84,2026-02-16,proj_11STWG6AokEni2F3eQugHb,,tokens,cus_11TqNEf2K2jxSUmeYyA1CX,req_11TvzyxfcabDS3LPgQq8ZP,2026-02-16 23:01:09.602,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,0,None,NaN,NaN,None
1,2026-02-16 23:01:31,85,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw115xgodiNUUviR6nuy,2026-02-16 23:01:24.874,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None
2,2026-02-16 23:01:31,86,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw117sHRJTd4yz5MhH47,2026-02-16 23:01:25.320,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None
3,2026-02-16 23:01:31,87,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw118euopY7ZiFXxBUgs,2026-02-16 23:01:25.504,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None
4,2026-02-16 23:01:31,88,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw119U2VTt8wSGAAAsYy,2026-02-16 23:01:25.694,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None
5,2026-02-16 23:01:31,89,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw11A9hhUvxENqWHEWg9,2026-02-16 23:01:25.854,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None
6,2026-02-16 23:01:31,90,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw11AzoSdMCvmhnhcLDJ,2026-02-16 23:01:26.052,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None
7,2026-02-16 23:01:31,91,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw11Bov8GcHUzwi1yV3C,2026-02-16 23:01:26.243,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None
8,2026-02-16 23:01:31,92,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw11CYpJj5viCbkmNPWL,2026-02-16 23:01:26.415,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None
9,2026-02-16 23:01:31,93,2026-02-16,proj_11STWG6AokEni2F3eQugHb,ent_11TqNEiMEehJjUMoxWAq7g,events,cus_11TqNEf2K2jxSUmeYyA1CX,req_11Tw11DMgcBVYwKfa9TVMR,2026-02-16 23:01:26.604,1,...,TXL,4055494417427953000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,987,None,NaN,NaN,None



=== lakehouse.metadata (showing up to 100 rows) ===


,__ingest_ts,id,event_date,project_id,customer_id,schema_version,timestamp,payload
0,2026-02-16 23:03:27,4055494417427953000,2026-02-16,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEDJVe8iPXHccWpZgK,1,2026-02-16 23:03:22.929,"""{\""resourceId\"":\""123\"",\""resourceType\"":\""us..."
1,2026-02-16 23:01:15,4055494417427953000,2026-02-16,proj_11STWG6AokEni2F3eQugHb,cus_11TqNEf2K2jxSUmeYyA1CX,1,2026-02-16 23:01:09.602,"""{\""resourceId\"":\""123\"",\""resourceType\"":\""us..."


In [31]:
from pyiceberg.expressions import And, EqualTo

# Load your table
table = catalog.load_table("lakehouse.verification")  # adjust namespace.table_name

# Filter by customer and date
scan = table.scan(
    row_filter=And(
        EqualTo("event_date", "2026-02-17"),
        EqualTo("customer_id", "cus_11TqNEDJVe8iPXHccWpZgK"),
    )
)

# Get all matching files
files = []
for task in scan.plan_files():
    files.append({
        "file_path": task.file.file_path,
        "record_count": task.file.record_count,
        "file_size_in_bytes": task.file.file_size_in_bytes,
    })

print(f"Found {len(files)} files with {sum(f['record_count'] for f in files)} total records")
for f in files:
    print(f["file_path"])


ValueError: Invalid timestamp without zone: 2026-02-17 (must be ISO-8601)

In [21]:
# 1. Create namespace
# catalog.create_namespace("default")

In [22]:
# 2. Load the pipeline-created table
verifications = catalog.load_table("lakehouse.verification")
print("Current Spec:", verifications.spec())
metadata = catalog.load_table("lakehouse.metadata")
print("Current Spec:", verifications.spec())
entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
print("Current Spec:", verifications.spec())
usage = catalog.load_table("lakehouse.usage")
print("Current Spec:", verifications.spec())
print(verifications.metadata.location)

Current Spec: [
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
  1003: date_partition: day(3)
]
Current Spec: [
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
  1003: date_partition: day(3)
]
Current Spec: [
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
  1003: date_partition: day(3)
]
Current Spec: [
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
  1003: date_partition: day(3)
]
s3://unprice-lakehouse-preview/__r2_data_catalog/019c68ba-c500-7d51-8df5-54250362d77e/019c68bc-5174-76f1-8ea6-20a563387470


In [24]:
# verifications
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.verification")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

New Spec: [
  1003: date_partition: day(3)
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
]

Partition Data Preview:


ValueError: Cannot get a snapshot as the table does not have any.

In [25]:
# usage
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.usage")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

New Spec: [
  1003: date_partition: day(3)
  1001: project_partition: identity(5)
  1002: customer_partition: identity(6)
]

Partition Data Preview:


ValueError: Cannot get a snapshot as the table does not have any.

In [26]:
# entitlement_snapshot
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.entitlement_snapshot")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

New Spec: [
  1003: date_partition: day(3)
  1001: project_partition: identity(4)
  1002: customer_partition: identity(5)
]

Partition Data Preview:


ValueError: Cannot get a snapshot as the table does not have any.

In [27]:
# metadata
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.metadata")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

New Spec: [
  1003: date_partition: day(3)
  1001: project_partition: identity(4)
  1002: customer_partition: identity(5)
]

Partition Data Preview:


ValueError: Cannot get a snapshot as the table does not have any.

In [28]:
# 4. Load the pipeline-created table
verifications = catalog.load_table("lakehouse.verification")
print("Current Spec:", verifications.spec())
metadata = catalog.load_table("lakehouse.metadata")
print("Current Spec:", verifications.spec())
entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
print("Current Spec:", verifications.spec())
usage = catalog.load_table("lakehouse.usage")
print("Current Spec:", verifications.spec())
print(verifications.metadata.location)

Current Spec: [
  1003: date_partition: day(3)
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
]
Current Spec: [
  1003: date_partition: day(3)
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
]
Current Spec: [
  1003: date_partition: day(3)
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
]
Current Spec: [
  1003: date_partition: day(3)
  1001: project_partition: identity(4)
  1002: customer_partition: identity(7)
]
s3://unprice-lakehouse-preview/__r2_data_catalog/019c68ba-c500-7d51-8df5-54250362d77e/019c68bc-5174-76f1-8ea6-20a563387470
